# Cobalt L-edge energy sweep: universal phase retrieval

Jointly reconstruct the fixed magnetic state from ideal CR (`+1`) and CL (`-1`) holograms at 20 energies. Every hologram is stretched about the detector center by $E/E_{\min}$ and center-cropped back to the original array size. Phase retrieval uses only the support mask stored at $E_{\min}$. The stored `xmcd_logs` are loaded only at the end for validation.

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage as ndi

# Make imports work when Jupyter starts in either the repository root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "library").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from library import phase_retrieval_universal as pr
from library import interactive


DATA_FILE= "/Users/riccardo/Data/cobalt_l_edge_energy_sweep.h5"

#if not DATA_FILE.exists():
#    raise FileNotFoundError(DATA_FILE)

In [ ]:


plt.rcParams.update({"figure.figsize": (7, 5), "image.cmap": "gray"})
print(DATA_FILE)

## Load the ideal holograms

Each energy group contains arrays with shape `(1, 512, 512)`. Squeezing and stacking produces one `(n_energy, ny, nx)` intensity stack per polarization.

In [ ]:
load_hologram="detected_holograms_without_beamstop"
load_hologram="ideal_holograms"



with h5py.File(DATA_FILE, "r") as handle:
    energies_eV = np.asarray(handle["energies"], dtype=float)
    group_names = sorted(handle[load_hologram].keys())

    if len(group_names) != len(energies_eV):
        raise ValueError("Number of ideal-hologram groups does not match energies")

    cr_ideal = np.stack([
        np.squeeze(np.asarray(handle[f"{load_hologram}/{name}/CR"], dtype=float))
        for name in group_names
    ])
    cl_ideal = np.stack([
        np.squeeze(np.asarray(handle[f"{load_hologram}/{name}/CL"], dtype=float))
        for name in group_names
    ])
    beamstop_masks = np.stack([
        np.squeeze(np.asarray(handle[f"beamstop_masks/{name}"], dtype=float))
        for name in group_names
    ])
    emin_index = int(np.argmin(energies_eV))
    emin_group = group_names[emin_index]
    supportmask = np.asarray(
        handle[f"supportmasks/{emin_group}"], dtype=float
    )

if cr_ideal.shape != cl_ideal.shape:
    raise ValueError(f"CR and CL shapes differ: {cr_ideal.shape} vs {cl_ideal.shape}")
if cr_ideal.shape[0] != len(energies_eV):
    raise ValueError("The hologram energy axis does not match energies")
if np.any(~np.isfinite(cr_ideal)) or np.any(~np.isfinite(cl_ideal)):
    raise ValueError("The ideal holograms contain NaN or infinite values")
if np.min(cr_ideal) < 0 or np.min(cl_ideal) < 0:
    raise ValueError("The ideal holograms must be non-negative intensities")

n_energy, ny, nx = cr_ideal.shape
print(f"energies: {energies_eV[0]:.1f} to {energies_eV[-1]:.1f} eV ({n_energy} points)")
print("CR stack:", cr_ideal.shape, cr_ideal.dtype)
print("CL stack:", cl_ideal.shape, cl_ideal.dtype)
print(f"Emin support: {emin_group} at {energies_eV[emin_index]:.1f} eV")

In [ ]:
interactive.cimshow(supportmask)

In [ ]:
%matplotlib widget

show_indices = np.unique(np.linspace(0, n_energy - 1, 3, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(len(show_indices)*3, 6), squeeze=False, sharex=True, sharey=True)
for column, index in enumerate(show_indices):
    for row, (stack, label) in enumerate(((cr_ideal, "CR"), (cl_ideal, "CL"))):
        axes[row, column].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(stack[index])))))
        axes[row, column].set_title(f"{label}, {energies_eV[index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Ideal hologram intensities, log display")
plt.tight_layout()

In [ ]:
%matplotlib widget

show_indices = np.unique(np.linspace(0, n_energy - 1, 3, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(len(show_indices)*3, 6), squeeze=False, sharex=True, sharey=True)
for column, index in enumerate(show_indices):
    for row, (stack, label) in enumerate(((cr_ideal, "CR"), (cl_ideal, "CL"))):
        axes[row, column].imshow(np.log1p(stack[index]))
        axes[row, column].set_title(f"{label}, {energies_eV[index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Ideal hologram intensities, log display")
plt.tight_layout()

## Normalize detector sampling and build metadata

The reciprocal-space scale changes with photon energy. For each energy $E$, the detector image is stretched about its center by $E/E_{\min}$ and sampled directly onto the original `(512, 512)` grid. This is equivalent to stretching followed by a centered crop, while avoiding a temporary larger array.

After that energy normalization, the optional `PRE_RETRIEVAL_CROP_SHAPE` and `PRE_RETRIEVAL_BIN_FACTOR` controls prepare smaller arrays for phase retrieval. Holograms and `mask_pixel` are cropped and binned in detector space in the same way. The support template is treated differently: detector cropping rescales the support to the new array shape so it occupies the same fraction of the image, while detector binning center-crops the support to the binned shape.

In [ ]:
def centered_rescale(image, scale, output_shape=None, order=1):
    """Rescale about the array center and return a fixed-size center crop."""
    image = np.asarray(image)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if scale <= 0 or not np.isfinite(scale):
        raise ValueError("scale must be positive and finite")
    if output_shape is None:
        output_shape = image.shape
    if np.isclose(scale, 1.0) and tuple(output_shape) == image.shape:
        return image.copy()

    input_center = (np.asarray(image.shape, dtype=float) - 1.0) / 2.0
    output_center = (np.asarray(output_shape, dtype=float) - 1.0) / 2.0
    matrix = np.eye(2) / float(scale)
    offset = input_center - matrix @ output_center
    return ndi.affine_transform(
        image,
        matrix=matrix,
        offset=offset,
        output_shape=tuple(output_shape),
        order=order,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )


def centered_resize(image, output_shape, order=1):
    """Resize a 2D image to output_shape while preserving relative position."""
    image = np.asarray(image)
    output_shape = tuple(int(value) for value in output_shape)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if len(output_shape) != 2 or min(output_shape) <= 0:
        raise ValueError("output_shape must contain two positive integers")
    if image.shape == output_shape:
        return image.copy()

    input_center = (np.asarray(image.shape, dtype=float) - 1.0) / 2.0
    output_center = (np.asarray(output_shape, dtype=float) - 1.0) / 2.0
    matrix = np.diag(np.asarray(image.shape, dtype=float) / np.asarray(output_shape, dtype=float))
    offset = input_center - matrix @ output_center
    return ndi.affine_transform(
        image,
        matrix=matrix,
        offset=offset,
        output_shape=output_shape,
        order=order,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )


def normalized_crop_shape(crop_shape, image_shape):
    """Return a validated crop shape or None when cropping is disabled."""
    if crop_shape is None:
        return None
    if isinstance(crop_shape, (int, np.integer)):
        crop_shape = (int(crop_shape), int(crop_shape))
    crop_shape = tuple(int(value) for value in crop_shape)
    if len(crop_shape) != 2 or min(crop_shape) <= 0:
        raise ValueError("PRE_RETRIEVAL_CROP_SHAPE must be None, an int, or (ny, nx)")
    if crop_shape[0] > image_shape[0] or crop_shape[1] > image_shape[1]:
        raise ValueError("PRE_RETRIEVAL_CROP_SHAPE cannot exceed the hologram shape")
    return crop_shape


def center_crop_2d(image, output_shape):
    """Center-crop a 2D image to output_shape."""
    image = np.asarray(image)
    output_shape = tuple(int(value) for value in output_shape)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if output_shape[0] > image.shape[0] or output_shape[1] > image.shape[1]:
        raise ValueError("Cannot center-crop to a larger shape")
    start_y = (image.shape[0] - output_shape[0]) // 2
    start_x = (image.shape[1] - output_shape[1]) // 2
    return image[start_y:start_y + output_shape[0], start_x:start_x + output_shape[1]].copy()


def center_crop_stack(stack, output_shape):
    """Center-crop the last two axes of a stack."""
    stack = np.asarray(stack)
    output_shape = tuple(int(value) for value in output_shape)
    if output_shape[0] > stack.shape[-2] or output_shape[1] > stack.shape[-1]:
        raise ValueError("Cannot center-crop to a larger shape")
    start_y = (stack.shape[-2] - output_shape[0]) // 2
    start_x = (stack.shape[-1] - output_shape[1]) // 2
    return stack[..., start_y:start_y + output_shape[0], start_x:start_x + output_shape[1]].copy()


def bin_stack(stack, factor, reducer="mean"):
    """Bin the last two axes of a stack after cropping to a multiple of factor."""
    factor = int(factor)
    if factor <= 0:
        raise ValueError("PRE_RETRIEVAL_BIN_FACTOR must be a positive integer")
    stack = np.asarray(stack)
    if factor == 1:
        return stack.copy()

    binned_shape = (stack.shape[-2] // factor, stack.shape[-1] // factor)
    if min(binned_shape) <= 0:
        raise ValueError("PRE_RETRIEVAL_BIN_FACTOR is too large for the image shape")
    crop_shape = (binned_shape[0] * factor, binned_shape[1] * factor)
    cropped = center_crop_stack(stack, crop_shape)
    reshaped = cropped.reshape(*cropped.shape[:-2], binned_shape[0], factor, binned_shape[1], factor)
    if reducer == "mean":
        return reshaped.mean(axis=(-3, -1))
    if reducer == "max":
        return reshaped.max(axis=(-3, -1))
    raise ValueError("reducer must be 'mean' or 'max'")


def build_support_template(support):
    """Build the shifted support template used for phase retrieval."""
    from skimage.draw import disk

    support = np.asarray(support) != 0
    structure = np.zeros((15, 15), dtype=bool)
    yy, xx = disk((structure.shape[0] // 2, structure.shape[1] // 2), 5)
    structure[yy, xx] = True

    shift_x = -(167 - support.shape[1] // 2)
    shift_y = -(345 - support.shape[0] // 2)
    rolled_dilated = np.roll(
        np.roll(ndi.binary_dilation(support, structure=structure), shift=shift_x, axis=1),
        shift=shift_y,
        axis=0,
    )
    rolled_support = np.roll(
        np.roll(support, shift=shift_x, axis=1),
        shift=shift_y,
        axis=0,
    )
    template = rolled_dilated.astype(bool)
    x_cut = min(300, template.shape[1])
    rolled_support=ndi.binary_erosion(rolled_support,iterations=3)
    template[:, :x_cut] = rolled_support[:, :x_cut]

    template=np.roll(np.roll(template, shift=-shift_x, axis=1), shift=-shift_y, axis=0)
   
    return template


def preprocess_detector_stack(stack, crop_shape, bin_factor, reducer="mean"):
    """Apply the detector-space crop/bin operations to holograms or masks."""
    prepared = np.asarray(stack).copy()
    if crop_shape is not None:
        prepared = center_crop_stack(prepared, crop_shape)
    if bin_factor > 1:
        prepared = bin_stack(prepared, bin_factor, reducer=reducer)
    return prepared


def preprocess_support_template(template, crop_shape, bin_factor, final_shape):
    """Apply the support-specific crop/bin rules requested for this notebook."""
    prepared = np.asarray(template) != 0
    if crop_shape is not None:
        # Cropping detector data changes the array shape. For the support, keep
        # the same fractional position and size instead of cutting pixels away.
        prepared = centered_resize(prepared.astype(float), crop_shape, order=0) > 0.5
    if bin_factor > 1:
        # Detector binning reduces the Fourier grid. For the support template,
        # keep the central real-space region with the final binned shape.
        prepared = center_crop_2d(prepared, final_shape) != 0
    return prepared.astype(bool)


# Optional phase-retrieval preprocessing controls.
# Examples: PRE_RETRIEVAL_CROP_SHAPE = (384, 384), PRE_RETRIEVAL_BIN_FACTOR = 2.
PRE_RETRIEVAL_CROP_SHAPE = None
PRE_RETRIEVAL_BIN_FACTOR = 1
USE_MASK_PIXEL = load_hologram != "ideal_holograms"

emin_eV = float(energies_eV[emin_index])
scale_factors = energies_eV / emin_eV
cr_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=1)
    for image, scale in zip(cr_ideal, scale_factors)
])
cl_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=1)
    for image, scale in zip(cl_ideal, scale_factors)
])
mask_pixel_energy = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=1)
    for image, scale in zip(beamstop_masks, scale_factors)
])

# Linear interpolation of non-negative inputs should remain non-negative;
# clip tiny floating-point undershoots defensively.
cr_rescaled = np.maximum(cr_rescaled, 0.0)
cl_rescaled = np.maximum(cl_rescaled, 0.0)
mask_pixel_energy = np.maximum(mask_pixel_energy, 0.0)

crop_shape = normalized_crop_shape(PRE_RETRIEVAL_CROP_SHAPE, (ny, nx))
bin_factor = int(PRE_RETRIEVAL_BIN_FACTOR)
supptemp_full = build_support_template(supportmask)

cr_prepared = preprocess_detector_stack(cr_rescaled, crop_shape, bin_factor, reducer="mean")
cl_prepared = preprocess_detector_stack(cl_rescaled, crop_shape, bin_factor, reducer="mean")
mask_pixel_energy = preprocess_detector_stack(mask_pixel_energy, crop_shape, bin_factor, reducer="max")
supptemp = preprocess_support_template(
    supptemp_full,
    crop_shape,
    bin_factor,
    final_shape=cr_prepared.shape[-2:],
)
ny_pr, nx_pr = cr_prepared.shape[-2:]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
axes[0].imshow(supportmask)
axes[0].set_title(f"Raw support: {emin_eV:.1f} eV")
axes[1].imshow(supptemp)
axes[1].set_title("Support for retrieval")
axes[2].imshow(np.log1p(cr_ideal[-1]))
axes[2].set_title(f"Raw CR at {energies_eV[-1]:.1f} eV")
axes[3].imshow(np.log1p(cr_prepared[-1]))
axes[3].set_title("Prepared CR for retrieval")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

print(f"Raw support pixels: {int(np.sum(supportmask != 0))} / {supportmask.size}")
print(f"Retrieval support pixels: {int(supptemp.sum())} / {supptemp.size}")
print(f"Scale-factor range: {scale_factors.min():.6f} to {scale_factors.max():.6f}")
print(f"Pre-retrieval crop shape: {crop_shape}")
print(f"Pre-retrieval bin factor: {bin_factor}")
print("Prepared stacks:", cr_prepared.shape, cl_prepared.shape)
print("Prepared mask_pixel:", mask_pixel_energy.shape, "use mask:", USE_MASK_PIXEL)

In [ ]:
# Interleave CR and CL so each adjacent pair belongs to the same energy.
holograms = np.stack([
    image
    for energy_index in range(n_energy)
    for image in (cr_prepared[energy_index], cl_prepared[energy_index])
])
mask_pixel = np.stack([
    image
    for energy_index in range(n_energy)
    for image in (mask_pixel_energy[energy_index], mask_pixel_energy[energy_index])
])
retrieval_mask_pixel = mask_pixel if USE_MASK_PIXEL else np.zeros_like(mask_pixel)
state_labels = ["fixed_state"] * (2 * n_energy)
energy_labels = np.repeat(energies_eV, 2)
polarizations = np.tile([+1.0, -1.0], n_energy)  # CR positive, CL negative
illumination_labels = ["fixed_beam"] * (2 * n_energy)

print("joint hologram stack:", holograms.shape)
print("joint mask stack:", retrieval_mask_pixel.shape)
print("support template:", supptemp.shape)
print("first four (energy, polarization):", list(zip(energy_labels, polarizations))[:4])

In [ ]:
%matplotlib widget
plt.close("all")

show_indices = np.unique(np.linspace(0, n_energy - 1, 3, dtype=int))
fig, axes = plt.subplots(3, len(show_indices), figsize=(len(show_indices) * 3, 7), sharex=True, sharey=True)
for column, energy_index in enumerate(show_indices):
    observation = 2 * energy_index
    label = "CR"
    axes[0, column].imshow(
         np.roll(np.roll(supptemp, shift=-39, axis=0), shift=-39, axis=1)*
         np.fft.fftshift(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(holograms[observation]))))),
        vmin=0,
        vmax=11,
    )
    axes[0, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
    axes[0, column].axis("off")
    axes[1, column].imshow(np.log1p(holograms[observation]))
    axes[1, column].set_title("Prepared hologram")
    axes[1, column].axis("off")
    axes[2, column].imshow(retrieval_mask_pixel[observation])
    axes[2, column].set_title("Prepared mask_pixel")
    axes[2, column].axis("off")
plt.suptitle("Prepared holograms, support, and mask")
plt.tight_layout()

In [ ]:
interactive.cimshow(supptemp)

## Joint phase retrieval

`QUICK_RUN=True` is a short end-to-end check. Set it to `False` for the longer starting recipe, then tune the iteration counts based on convergence. The physical model is

$$L(E,p)=C+q_c(E)+p\,q_m(E)m_z,$$

with one shared state and beam. No ground-truth arrays are passed to the reconstruction.

In [ ]:
QUICK_RUN = False

if QUICK_RUN:
    inner_Nit = [5, 1]
    outer_iterations = 5
    warmup_Nit = [20, 5]
    physical_iterations = 3
else:
    inner_Nit = [700,50]
    outer_iterations = 5
    warmup_Nit = [700, 50]
    physical_iterations = 5

recipe = {
    "projection_model": "svd",#"physical_factorized",
    "inner_mode": ["HAPRE","ER"],
    "inner_Nit": inner_Nit,
    "outer_iterations": outer_iterations,
    "warmup_mode": ["HAPRE", "ER"],
    "warmup_Nit": warmup_Nit,
    "beta_zero": 0.5,
    "beta_mode": "arctan",
    "alpha_zero": 0.,
    "TV_freq": 1e9,
    "alpha_mode": "linear_to_0",
    "average_img": 1,
    "plot_every": 20,
    "shuffle_observations": True,
    "random_seed": 7,
    "projection_every": 1,
    "projection_start": None,
    "projection_relaxation": 1.0,
    "physical_iterations": physical_iterations,
    "energy_values": energies_eV,
    "charge_spectral_constraint": "free",
    "magnetic_spectral_constraint": "free",
    "final_fourier_constraint": False,
    "zero_magnetization_outside_support": False,
    "projection_constraints_inside_support_only": False,
}
fields, fieldswarmup,components, bsmasks, errors = pr.universal_phase_retrieval_algorithm(
    holograms,
    retrieval_mask_pixel*0,
    supptemp,
    state_labels=state_labels,
    energy_labels=energy_labels,
    polarization_coefficients=polarizations,
    illumination_labels=illumination_labels,
    saturated_states=None,
    universal_recipe=recipe,
)

try:
    print(f"fit residual RMS: {components['fit_residual_rms']:.4g}")
    print("magnetic scale anchored:", components["magnetic_scale_anchored"])
    print("fully identifiable:", components["identifiable"])
    print(f"runtime: {errors['runtime_seconds']:.1f} s")
except: pass

With one unknown state, the product $q_m(E)m_z$ is identifiable but its factors have a scale/sign ambiguity. The library clips the recovered reduced magnetization to `[-1, 1]`. A known saturated state or an absolutely calibrated magnetic spectrum is required to anchor the physical scale.

In [ ]:
interactive.cimshow(np.abs(fields))

In [ ]:
import scipy
label, N=scipy.ndimage.label(supptemp)
counts = np.bincount(label[label!=0])
most_frequent_value = np.argmax(counts)
y0,x0=scipy.ndimage.center_of_mass(label, most_frequent_value)
print(most_frequent_value,y0,x0)
y0,x0=int(y0),int(x0)
r=int(np.sqrt(np.sum(label==most_frequent_value)/np.pi)*1.)
roi=np.s_[np.clip(y0-r,0,None):np.clip(y0+r,0,supptemp.shape[0]-1),np.clip(x0-r,0,None):np.clip(x0+r,0,supptemp.shape[0]-1)]

print(roi)

In [ ]:
%matplotlib widget
plt.close("all")
interactive.cimshow(np.abs(supptemp)[roi])

In [ ]:
magnetization = components["magnetization_by_state"]["fixed_state"]
charge_response = np.asarray(components["charge_response"])
magnetic_response = np.asarray(components["magnetic_response"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
image = axes[0].imshow(magnetization[roi], cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Recovered magnetic state $m_z$")
axes[0].axis("off")
fig.colorbar(image, ax=axes[0], fraction=0.046)

axes[1].plot(energies_eV, np.abs(charge_response), "o-", label="abs")
ax2=axes[1].twinx()
ax2.plot(energies_eV, 2*np.pi*((np.angle(charge_response)/(2*np.pi) +6)%2), "o-", label="phase", color="orange")
axes[1].set_title("Recovered charge response")
axes[1].set_xlabel("Energy (eV)")
axes[1].legend()

axes[2].plot(energies_eV, np.abs(magnetic_response), "o-", label="abs")
ax3=axes[2].twinx()
ax3.plot(energies_eV, np.angle(magnetic_response), "o-", label="phase", color="orange")
axes[2].set_title("Recovered magnetic response")
axes[2].set_xlabel("Energy (eV)")
axes[2].legend()
plt.tight_layout()

## Validate against the simulated XMCD state

The complex `xmcd_logs` obey `log(CR/CL) = 2 q_m(E)m_z`. Because stretching reciprocal-space data by $E/E_{\min}$ contracts the corresponding real-space coordinates by $E_{\min}/E$, each validation map is normalized by that inverse factor before the rank-one decomposition. This validation data was not used by phase retrieval.

In [ ]:
with h5py.File(DATA_FILE, "r") as handle:
    true_xmcd_logs = np.stack([
        np.squeeze(np.asarray(handle[f"xmcd_logs/{name}"]))
        for name in group_names
    ])

true_xmcd_logs = np.stack([
    centered_rescale(log_map, 1.0 / scale, output_shape=(ny, nx), order=1)
    for log_map, scale in zip(true_xmcd_logs, scale_factors)
])
if crop_shape is not None:
    true_xmcd_logs = np.stack([
        centered_resize(log_map, crop_shape, order=1)
        for log_map in true_xmcd_logs
    ])
if bin_factor > 1:
    true_xmcd_logs = center_crop_stack(true_xmcd_logs, holograms.shape[-2:])

# The first spatial right-singular vector is the common state times an
# arbitrary complex phase. Rotate it to be maximally real, then normalize.
truth_ny, truth_nx = true_xmcd_logs.shape[-2:]
_, _, vh = np.linalg.svd(true_xmcd_logs.reshape(n_energy, -1), full_matrices=False)
truth_complex = vh[0].reshape(truth_ny, truth_nx)
truth_phase = 0.5 * np.angle(np.sum(truth_complex ** 2))
truth_state = np.real(truth_complex * np.exp(-1j * truth_phase))
truth_state /= np.max(np.abs(truth_state))

# The component magnetization is in the log-object frame. The support template
# passed to phase retrieval is in the support-mask frame, so shift it here.
inside = np.fft.fftshift(supptemp.astype(bool))
if np.sum(magnetization[inside] * truth_state[inside]) < 0:
    truth_state *= -1

correlation = np.corrcoef(magnetization[inside], truth_state[inside])[0, 1]
difference = magnetization - truth_state

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, image_data, title in zip(
    axes,
    (magnetization, truth_state, difference),
    ("Phase-retrieval state", "Simulated XMCD state", "Difference"),
):
    image = axis.imshow(image_data, cmap="RdBu_r", vmin=-1, vmax=1)
    axis.set_title(title)
    axis.axis("off")
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.suptitle(f"State correlation inside support: {correlation:.4f}")
plt.tight_layout()

print(f"state correlation inside support: {correlation:.6f}")

In [ ]:
%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fields)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(2*len(show_indices), 4), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.angle(exit_waves[2 * energy_index + offset])
        axes[row, column].imshow(np.fft.fftshift(phase)[roi], cmap="twilight", vmin=-np.pi, vmax=np.pi)
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        #axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

In [ ]:
%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fieldswarmup)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(2*len(show_indices), 4), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.imag(exit_waves[2 * energy_index + offset]-offset*exit_waves[2 * energy_index + 0])
        axes[row, column].imshow((np.fft.fftshift(phase)*supptemp))
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fields)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(2*len(show_indices), 4), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.imag(exit_waves[2 * energy_index + offset]-offset*exit_waves[2 * energy_index + 0])
        axes[row, column].imshow((np.fft.fftshift(phase)*supptemp))
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

In [ ]:
fig,ax=plt.subplots(2,2)
ax[0,0].plot(np.abs(components["charge_response"]))
ax[0,1].plot(np.abs(components["magnetic_response"]))
ax[1,0].plot(2*np.pi*(np.angle(components["charge_response"]/(2*np.pi) +6)%1))
ax[1,0].plot(2*np.pi*(np.angle(components["charge_response"]/(2*np.pi) +6)%1)-2*np.pi)
ax[1,1].plot(2*np.pi*((np.angle(components["magnetic_response"]/(2*np.pi) +6)%1)))
ax[1,1].plot(2*np.pi*((np.angle(components["magnetic_response"]/(2*np.pi) +6)%1))-2*np.pi)

In [ ]:
plt.close("all")
%matplotlib inline
fig,ax=plt.subplots()
temp=(np.real(supptemp*np.fft.fftshift(components["magnetization"][0]))[roi])
mi,ma=np.percentile(temp,  (2,98))
ax.imshow(temp, vmin=mi, vmax=ma)

In [ ]:
components["magnetization"]